# Fine-Grained Mixture-of-Experts (MoE) & Dynamic Routing

## The MoE Routing Bottleneck

### 1. Routing Collapse (The "Rich Get Richer" Effect)

A standard Top-K gate routes tokens to experts with the highest initial affinity scores: $G(x) = \text{TopK}(\text{Softmax}(W_g \cdot x))$.

Early in training, 2 out of 8 experts receive slightly better gradients. The gate over-routes tokens to those 2 experts, leaving the other 6 experts untrained (under-utilized). The model collapses back to a small dense model.

### 2. The Auxiliary Loss Trap (The Classical Solution)

To prevent collapse, legacy architectures add an "Auxiliary Load Balancing Loss" ($L_{\text{aux}}$) to force equal token distribution across experts.

Problem: $L_{\text{aux}}$ conflicts with the primary Cross-Entropy loss. It forces tokens to go to non-optimal experts just to satisfy a uniform quota.

Result: Degraded task accuracy and sub-optimal expert specialization.

## 2. The DeepSeekMoE Paradigm Shift

DeepSeekMoE (pioneered in DeepSeek-V2/V3) introduced two architectural innovations to solve these bottlenecks.

### DeepSeek-V3 MoE Layer Architecture

```text
[Token Input]
    ↓
[Shared Experts]                    [Routed Experts]
(Always Active)                     (Fine-Grained & Dynamic)
    ├─ Global domain knowledge      ├─ 256 Total Fine-Grained Experts
    ├─ Prevents redundancy          ├─ 8 Experts Activated per Token
    └─ Common patterns              └─ Dynamic routing (Auxiliary-Loss-Free)
```

### Innovation A: Fine-Grained Expert Granularity

In legacy MoE (e.g., Mixtral 8x7B), there are $N=8$ large experts, and Top-2 are selected.

DeepSeekMoE splits experts into much smaller units:

- Instead of $N$ experts with hidden dimension $D_{\text{hidden}}$, it creates $m \times N$ experts with hidden dimension $D_{\text{hidden}} / m$.
- Instead of activating 2 large experts, it activates $m \times 2$ fine-grained experts.

Benefit: Tokens can combine fine-grained capabilities in more combinatorial ways without increasing total active parameters or compute FLOPs.

### Innovation B: Shared Experts

DeepSeekMoE isolates a subset of experts as Shared Experts. These shared experts are always active for every single token.

Benefit: Common linguistic patterns, punctuation rules, and general syntax are processed by shared experts. This frees up the routed experts to focus strictly on specialized domain knowledge (math, code, specific languages).

## 3. Mathematical Formulation: Auxiliary-Loss-Free Load Balancing

DeepSeek-V3 completely removes the accuracy-degrading auxiliary loss $L_{\text{aux}}$. Instead, it uses Dynamic Bias Correction during gating.

### Step 1: Affinity Score Calculation

For an input token representation $u_t \in \mathbb{R}^D$, we compute raw affinity scores $s_{i,t}$ across all $N$ routed experts using routing weights $W_g \in \mathbb{R}^{N \times D}$:

$$
s_{i,t} = W_{g, i} \cdot u_t
$$

### Step 2: Dynamic Bias Adjustment (Auxiliary-Loss-Free)

Instead of forcing $W_g$ to update via an auxiliary loss, a dynamic bias term $b_i$ is added only during expert selection:

$$
\text{Selection Score: } \hat{s}_{i,t} = s_{i,t} + b_i
$$

$$
\text{Selected Experts: } \mathcal{E}_t = \text{TopK}\left( \{\hat{s}_{i,t}\}_{i=1}^N, \, K \right)
$$

If expert $i$ is overloaded (receiving more tokens than average), its bias $b_i$ is decreased by a step size $\gamma$.

If expert $i$ is underloaded (receiving fewer tokens than average), its bias $b_i$ is increased by $\gamma$.

$$
b_i \leftarrow \begin{cases} b_i - \gamma & \text{if Expert } i \text{ is overloaded} \\ b_i + \gamma & \text{if Expert } i \text{ is underloaded} \end{cases}
$$

### Step 3: Output Gating Weight Calculation

Once the top-$K$ experts $\mathcal{E}_t$ are selected using the biased scores $\hat{s}_{i,t}$, the actual softmax routing weights $g_{i,t}$ applied to the expert outputs use the original unbiased affinity scores $s_{i,t}$:

$$
g_{i,t} = \frac{\exp(s_{i,t})}{\sum_{j \in \mathcal{E}_t} \exp(s_{j,t})}, \quad \forall i \in \mathcal{E}_t
$$

### Step 4: Final MoE Layer Output

The final token representation combining Shared Experts ($\text{FFN}_{\text{shared}}$) and Routed Experts ($\text{FFN}_{i}$) is expressed as:

$$
y_t = \underbrace{\text{FFN}_{\text{shared}}(u_t)}_{\text{Always Active}} + \sum_{i \in \mathcal{E}_t} g_{i,t} \cdot \text{FFN}_{i}(u_t)
$$

## 4. Architectural Comparison Table

| Metric / Mechanism | Legacy MoE (GShard / Switch) | Mixtral 8x7B | DeepSeek-V3 MoE |
| --- | --- | --- | --- |
| Total Experts ($N$) | 16–64 Large Experts | 8 Large Experts | 256 Fine-Grained Experts |
| Activated Experts ($K$) | Top-1 or Top-2 | Top-2 | Top-8 Fine-Grained Experts |
| Shared Experts | None | None | 1 Dedicated Shared Expert |
| Load Balancing | Auxiliary Loss $L_{\text{aux}}$ | Auxiliary Loss $L_{\text{aux}}$ | Auxiliary-Loss-Free Dynamic Bias ($b_i$) |
| Total vs Active Params | e.g., 137B Total / 22B Active | 46.7B Total / 12.9B Active | 671B Total / 37B Active |

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set seed for reproducible expert routing checks
torch.manual_seed(42)

# =====================================================================
# 1. FINE-GRAINED FEED-FORWARD EXPERT NETWORK
# =====================================================================

class FineGrainedExpert(nn.Module):
    """
    Individual fine-grained SwiGLU / GELU Feed-Forward Network.
    DeepSeek-V3 uses smaller intermediate hidden dimensions to create 
    more granular experts without inflating overall compute.
    """
    def __init__(self, d_model: int, d_expert_hidden: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_expert_hidden, bias=False)
        self.w2 = nn.Linear(d_expert_hidden, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_expert_hidden, bias=False) # SwiGLU gate

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # SwiGLU Activation: (Swish(W1 * x) * W3 * x) -> W2
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


# =====================================================================
# 2. DEEPSEEK-V3 AUXILIARY-LOSS-FREE MOE ROUTER
# =====================================================================

class DeepSeekRouter(nn.Module):
    """
    DeepSeek-V3 Dynamic Router with Auxiliary-Loss-Free Load Balancing.
    Maintains a dynamic bias vector (b_i) updated on the fly to equalize
    expert utilization without corrupting primary gradients.
    """
    def __init__(self, d_model: int, num_routed_experts: int, top_k: int, bias_lr: float = 0.001):
        super().__init__()
        self.num_experts = num_routed_experts
        self.top_k = top_k
        self.bias_lr = bias_lr

        # Linear projection to compute unnormalized affinity scores: W_g * x
        self.weight = nn.Parameter(torch.randn(num_routed_experts, d_model) * 0.02)
        
        # Dynamic Load-Balancing Bias b_i (Not updated by backprop loss, adjusted manually)
        self.register_buffer("expert_bias", torch.zeros(num_routed_experts))

    def forward(self, x: torch.Tensor):
        """
        x: [Batch * Seq_Len, d_model]
        Returns:
            topk_indices: [Batch * Seq_Len, top_k] - Selected expert IDs per token
            routing_weights: [Batch * Seq_Len, top_k] - Normalized Softmax weights
        """
        N_tokens, D = x.shape

        # Step 1: Compute raw affinity scores s_{i, t} = W_g * u_t
        raw_scores = F.linear(x, self.weight) # [N_tokens, num_experts]

        # Step 2: Apply dynamic bias ONLY for Top-K selection: \hat{s} = s + b
        selection_scores = raw_scores + self.expert_bias.unsqueeze(0)

        # Step 3: Select Top-K fine-grained experts
        _, topk_indices = torch.topk(selection_scores, self.top_k, dim=-1) # [N_tokens, top_k]

        # Step 4: Compute gating weights using ORIGINAL UNBIASED scores s_{i, t}
        # Gather original scores for selected experts
        selected_raw_scores = torch.gather(raw_scores, dim=-1, index=topk_indices) # [N_tokens, top_k]
        
        # Apply Softmax across the top-k selected experts
        routing_weights = F.softmax(selected_raw_scores, dim=-1) # [N_tokens, top_k]

        # Step 5: Update dynamic bias b_i for Auxiliary-Loss-Free load balancing
        if self.training:
            with torch.no_grad():
                # Count how many tokens were routed to each expert in this batch
                tokens_per_expert = torch.bincount(
                    topk_indices.view(-1), minlength=self.num_experts
                ).float()
                
                target_tokens = (N_tokens * self.top_k) / self.num_experts
                
                # Overloaded experts (count > target) -> Decrease bias
                # Underloaded experts (count < target) -> Increase bias
                load_error = target_tokens - tokens_per_expert
                self.expert_bias += self.bias_lr * torch.sign(load_error)

        return topk_indices, routing_weights


# =====================================================================
# 3. DEEPSEEK-V3 MOE LAYER (SHARED + ROUTED EXPERTS)
# =====================================================================

class DeepSeekMoELayer(nn.Module):
    def __init__(
        self, 
        d_model: int = 128, 
        d_expert_hidden: int = 32, 
        num_routed_experts: int = 16, 
        top_k: int = 4, 
        num_shared_experts: int = 1
    ):
        super().__init__()
        self.d_model = d_model
        self.num_routed = num_routed_experts
        self.top_k = top_k

        # 1. Shared Experts (Always active for every token)
        self.shared_experts = nn.ModuleList([
            FineGrainedExpert(d_model, d_expert_hidden * num_shared_experts)
        ])

        # 2. Fine-Grained Routed Experts
        self.routed_experts = nn.ModuleList([
            FineGrainedExpert(d_model, d_expert_hidden) for _ in range(num_routed_experts)
        ])

        # 3. Auxiliary-Loss-Free Router
        self.router = DeepSeekRouter(d_model, num_routed_experts, top_k)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [Batch, Seq_Len, d_model]
        """
        B, T, D = x.shape
        x_flat = x.view(-1, D) # Flatten to [B * T, d_model] for token-level routing
        N_tokens = x_flat.shape[0]

        # -------------------------------------------------------------
        # BRANCH A: SHARED EXPERT EXECUTION (Always Active)
        # -------------------------------------------------------------
        out_shared = torch.zeros_like(x_flat)
        for shared_exp in self.shared_experts:
            out_shared += shared_exp(x_flat)

        # -------------------------------------------------------------
        # BRANCH B: ROUTED FINE-GRAINED EXPERT EXECUTION
        # -------------------------------------------------------------
        topk_indices, routing_weights = self.router(x_flat) # [N_tokens, top_k]

        out_routed = torch.zeros_like(x_flat)

        # Vectorized dispatch loop over experts
        for exp_id in range(self.num_routed):
            # Find tokens assigned to current expert `exp_id`
            # mask shape: [N_tokens, top_k]
            mask = (topk_indices == exp_id)
            
            if not mask.any():
                continue # Skip unselected experts in this step

            # Extract token and top-k position indices where current expert was chosen
            token_idx, k_idx = torch.where(mask)

            # Get token representations
            selected_tokens = x_flat[token_idx] # [Num_Assigned, d_model]
            
            # Pass through fine-grained expert
            expert_output = self.routed_experts[exp_id](selected_tokens)
            
            # Get gating weights for these specific assignments
            weights = routing_weights[token_idx, k_idx].unsqueeze(-1) # [Num_Assigned, 1]

            # Accumulate weighted output back into output tensor
            out_routed.index_add_(0, token_idx, expert_output * weights)

        # Combine Shared Experts and Routed Experts
        final_out = out_shared + out_routed
        return final_out.view(B, T, D)


# =====================================================================
# 4. UNIT TEST & ROUTING PROFILE VERIFICATION
# =====================================================================

if __name__ == "__main__":
    bsz, seq_len, d_model = 4, 32, 128
    num_routed_experts = 16
    top_k = 4

    moe_layer = DeepSeekMoELayer(
        d_model=d_model,
        d_expert_hidden=32,
        num_routed_experts=num_routed_experts,
        top_k=top_k,
        num_shared_experts=1
    )

    dummy_input = torch.randn(bsz, seq_len, d_model)

    print("--- Executing DeepSeekMoE Forward Pass ---")
    out = moe_layer(dummy_input)
    print(f"Input Shape : {dummy_input.shape}")
    print(f"Output Shape: {out.shape}")

    # Simulated Training Run to Observe Load-Balancing Bias Updates
    print("\n--- Simulating 100 Training Steps to Verify Load Balancing ---")
    moe_layer.train()
    optimizer = torch.optim.AdamW(moe_layer.parameters(), lr=1e-3)

    for step in range(1, 101):
        x = torch.randn(bsz, seq_len, d_model)
        optimizer.zero_grad()
        output = moe_layer(x)
        loss = output.sum() # Dummy loss for iteration
        loss.backward()
        optimizer.step()

    print("\n✅ Final Dynamic Bias Values (b_i) across 16 Routed Experts:")
    print(moe_layer.router.expert_bias.cpu().numpy().round(4))

--- Executing DeepSeekMoE Forward Pass ---
Input Shape : torch.Size([4, 32, 128])
Output Shape: torch.Size([4, 32, 128])

--- Simulating 100 Training Steps to Verify Load Balancing ---

✅ Final Dynamic Bias Values (b_i) across 16 Routed Experts:
[ 0.086 -0.    -0.069  0.053 -0.064 -0.047 -0.001 -0.011 -0.035  0.026
  0.035 -0.049 -0.034  0.001  0.083 -0.059]
